# Cheating picoseconds

In [1]:
import numpy as np

In [6]:
with open("inputs/20", "r") as fp:
    data = fp.read()[:-1]

In [17]:
def process_puzzle(data):
    """Process input puzzle.
    :rparam: (Matrix: -1 = wall, 0 = corridor, start_position, end_position)
    """
    lines = data.split("\n")
    l0 = len(lines)
    l1 = len(lines[0])
    M = np.zeros((l0, l1))
    p_start = (-1, -1)
    p_end = (-1, -1)
    
    for idx, line in enumerate(lines):
        for jdx, c in enumerate(line):
            if c == "#":
                M[idx, jdx] = -1
            elif c == "E":
                p_end = (idx, jdx)
            elif c == "S":
                p_start = (idx, jdx)
                
    return M, p_start, p_end

In [18]:
M, p_start, p_end = process_puzzle(data)

# Part 1

1. Calculate "best time" without cheating
2. Calculate all cheat time


## Calculate normal time

In [30]:
def add_postion(p, dd):
    a, b = p
    a1, b1 = dd 
    return (a+a1, b+b1)

In [55]:
def get_time_matrix(M, p_start, p_end):
    # Init
    M_time = M.copy()
    maxi = np.prod(M_time.shape)
    M_time[M_time == 0] = maxi

    # Loop 
    lst = [(p_start, 0)]
    while lst != []:
        p, s = lst.pop(0) # Leave first element
        
        if M_time[p] <= s:
            # If already optimal, no need to process twice
            continue
    
        # Update score
        M_time[p] = s
        for dd in [(0, 1), (0, -1), (-1, 0), (1, 0)]:
            p1 = add_postion(p, dd)
            if M_time[p1] == -1:
                continue
            else:
                lst.append((p1, s+1))
        
    return M_time    

In [56]:
M_time = get_time_matrix(M, p_start, p_end)
M_time1 = get_time_matrix(M, p_end, p_start)


### Get best time

In [57]:
time_best = M_time[p_end]

In [58]:
print(time_best)

9408.0


In [60]:
M_glob = (M_time + M_time1)

In [63]:
print((M_glob == time_best).sum())

9409


OK, there is a single unique path from start to end

In [69]:
print((M_glob > time_best).sum())
print(((M_glob < time_best) & (M_glob > 0)).sum())


0
0


## Compute cheated times

Procedure: use the `M_time` matrix. 

Check all possible moves:

- UU | LL | RR | DD
- UD | UR | xx

Use the matrix to see if we get something interesting

In [71]:
def check_p1(p, M):
    l0, l1 = M.shape
    x, y = p
    if (x < 0) | (y < 0) | (x >= l0) | (y >= l1):
        return False
    else:
        return True

In [93]:
moves = [
    (0, 2), (0, -2), (-2, 0), (2, 0), # "classical moves"
    (1, 1), (1, -1), (-1, -1), (-1, 1)
]

cnt = 0
for p in np.array(np.where(M_glob == time_best)).T:
    p = (int(p[0]), int(p[1]))
    s = M_time[p]
    for move in moves:
        p1 = add_postion(p, move)
        if check_p1(p1, M):
            s1 = M_time[p1]
            ds = s1 - s
            if ds >= 102: # 100 + 2 because it still require to move
                cnt += 1

print(cnt)

1365


# Part 2: Cheat last longuer

#### Gather the list of moves

In [98]:
lst_cheats = []
for idx in range(-20, 21, 1):
    for jdx in range(-20, 21, 1):
        timing = abs(idx) + abs(jdx)
        if (timing <= 20) & (timing > 1):
            # Condition 
            lst_cheats.append(((idx, jdx), timing))
            
            
        

### Evaluate

In [100]:

cnt = 0
for p in np.array(np.where(M_glob == time_best)).T:
    p = (int(p[0]), int(p[1]))
    s = M_time[p]
    for move, t1 in lst_cheats:
        p1 = add_postion(p, move)
        if check_p1(p1, M):
            s1 = M_time[p1]
            ds = s1 - s
            if ds >= 100 + t1: # 100 + 2 because it still require to move
                cnt += 1

print(cnt)

986082
